In [37]:

%pip install python-dotenv --upgrade --quiet langchain langchain-groq


In [38]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")



In [39]:
MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Provide debugging steps, code fixes, and technical explanations.
Avoid unnecessary fluff."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Expert.
Be empathetic and professional.
Focus on refunds, charges, invoices, subscriptions, and policies.
Explain steps clearly and politely."""
    },
    "general": {
        "system_prompt": """You are a helpful general assistant.
Handle casual conversation and general inquiries politely."""
    }

}


In [40]:
def route_prompt(user_input):
    router_prompt = f"""
Classify the following text into one of these categories:
[technical, billing, general, tool]

If the user is asking for real-time data like cryptocurrency prices,
current stock values, or live information, classify as "tool".

Return ONLY the category name. No explanation.

Text:

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": router_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category


In [41]:
def get_bitcoin_price():
    # Mock API call
    return "The current price of Bitcoin is $64,250 (mock data)."


In [42]:
def process_request(user_input):
    category = route_prompt(user_input)

    # TOOL ROUTING
    if category == "tool":
        if "bitcoin" in user_input.lower():
            print("🔎 Routed to: TOOL expert\n")
            return get_bitcoin_price()
        else:
            return "Tool category detected but no tool available."

    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    print(f"🔎 Routed to: {category.upper()} expert\n")
    return response.choices[0].message.content.strip()


In [43]:
print(process_request("My python script is throwing an IndexError on line 5."))


🔎 Routed to: TECHNICAL expert

To troubleshoot the `IndexError` on line 5, I need more information about your script. Please provide the following:

1. The code snippet that's causing the error (line 5 and surrounding code).
2. The complete error message.
3. The data or input that's causing the error.

However, if you're dealing with lists and indexing, I can provide a general guide:

**Common causes of IndexError:**

1. Out-of-range indexing (e.g., `my_list[10]` when `my_list` only has 10 elements).
2. Using `append` instead of `extend` when adding elements to a list.

**Basic debugging steps:**

1. Print the list or data structure before line 5 to ensure it has the expected elements.
2. Check the indexing (e.g., `my_list[0]` instead of `my_list[10]`).
3. Verify that the list is not empty or None.

**Example:**

```python
my_list = [1, 2, 3]  # Ensure this list has 3 elements

try:
    print(my_list[3])  # Attempting to access an out-of-range index
except IndexError as e:
    print(f"

In [44]:
print(process_request("I was charged twice for my subscription this month."))


🔎 Routed to: BILLING expert

I'm so sorry to hear that you were charged twice for your subscription this month. I'm here to help you resolve this issue as quickly as possible.

To investigate this further, could you please provide me with some information? 

1. What is your subscription name or the name of the product/service you're subscribed to?
2. The date of the first charge and the date of the second charge.
3. The amounts of the two charges.
4. Your payment method (credit card, bank transfer, etc.)?

Once I have this information, I'll review your account and look into the cause of the duplicate charge. I'll then guide you through the steps to correct the issue and potentially issue a refund for the excess amount charged.

Please know that I'm here to help, and I'll do my best to turn this around for you as soon as possible.


In [45]:
print(process_request("Tell me a fun fact about space."))


🔎 Routed to: GENERAL expert

That sounds like a blast! Here's a fun fact about space: Did you know that there is a giant storm on Jupiter that has been raging for at least 187 years? It's called the Great Red Spot, and it's a persistent anticyclonic storm that's larger than Earth in diameter. The storm is so massive that it could swallow several Earths whole. Isn't that mind-blowing?


In [46]:
print(process_request("What is the current price of Bitcoin?"))


🔎 Routed to: TOOL expert

The current price of Bitcoin is $64,250 (mock data).
